<img src="https://www.luxonis.com/logo.svg" width="400">

# Conversion of RF-DETR to RVC4

## 🌟 Overview

This tutorial shows how to convert **RF-DETR** models for deployment on **RVC4** devices using the Luxonis toolchain. The example uses **RF-DETR Nano**, but the same approach can be applied to other RF-DETR variants as well.

We'll export the model to ONNX, package it as a Luxonis **NN Archive**, convert it to **RVC4** with HubAI, and run inference with DepthAI.

Before export, a small compatibility adjustment is needed for the RVC4 conversion pipeline. Rather than modifying the RF-DETR source code, this tutorial performs the adjustment at runtime by replacing selected PyTorch module implementations on the live model object.

> **Note:** This deployment workflow targets **RVC4 devices only**.

In this tutorial, we will:

1. Install the required Python packages.
2. Clone RF-DETR at a known release.
3. Define RVC4-compatible module implementations inside the notebook.
4. Inject those modules into the live model before export.
5. Export RF-DETR to ONNX.
6. Build a Luxonis NN Archive with `ArchiveGenerator`.
7. Convert the NN Archive to RVC4 with HubAI.
8. Run a minimal DepthAI inference pipeline.

This tutorial uses:

* RF-DETR `1.7.1`
* RF-DETR Nano
* ONNX opset `17`
* HubAI RVC4 conversion with `FP16_STANDARD` precision and SNPE `2.41.0`


## 📜 Table of Contents

- 🛠️ [Installation](#installation)
- 🧬 [Clone RF-DETR](#clone-rf-detr)
- 🧩 [Runtime Class Injection](#runtime-class-injection)
- 📤 [Export RF-DETR to ONNX](#export-rfdetr-to-onnx)
- 📦 [Create the NN Archive](#create-the-nn-archive)
- 🤖 [Convert the NN Archive to RVC4](#convert-the-nn-archive-to-rvc4)
- 📷 [DepthAI Script](#depthai-script)


<a name="colab-notes"></a>

## ☁️ Colab Notes

If you run this notebook in Google Colab, you can complete the export, NN Archive creation, and HubAI conversion steps there. The final DepthAI runtime step should be run on a local machine that can access an RVC4 device.


<a name="installation"></a>

## 🛠️ Installation

Install the packages needed for ONNX export, ONNX inspection, NN Archive creation, HubAI conversion, and DepthAI runtime.

In [ ]:
%pip install -q luxonis-ml==0.8.6 onnx==1.21.0 onnxruntime==1.23.2 depthai==3.7.1 depthai-nodes==0.5.1
%pip install -q -U hubai-sdk
%pip install -q numpy==2.0.2

<a name="local-files"></a>

## 🗂️ Local Files

The notebook will create:

```text
onnx_model/
rf-detr/
rfdetr-nano-onnx.tar.xz
rfdetr-nano-onnx.rvc4.tar.xz
rvc4_class_injection.py
```

The `rf-detr/` directory is the cloned upstream repository. The `rvc4_class_injection.py` file is created by the notebook itself and contains the local module used for export-time class injection.


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
ONNX_DIR = ROOT / "onnx_model"
ONNX_DIR.mkdir(exist_ok=True)

print("Working directory:", ROOT.resolve())
print("ONNX directory:", ONNX_DIR.resolve())


<a name="clone-rf-detr"></a>

## 🧬 Clone RF-DETR

Clone RF-DETR and check out the release used by this tutorial.

This tutorial uses `1.7.1`, which was validated with short commit hash `6e1620e`.

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("rf-detr")

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/roboflow/rf-detr.git", str(repo_dir)],
        check=True,
    )

for command in [
    ["git", "fetch", "--all", "--tags"],
    ["git", "checkout", "1.7.1"],
    ["git", "rev-parse", "--short", "HEAD"],
    ["git", "describe", "--tags", "--always"],
    ["git", "status", "--short"],
]:
    print("$", " ".join(command))
    result = subprocess.run(
        command,
        cwd=repo_dir,
        check=True,
        text=True,
        capture_output=True,
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())


Install RF-DETR in editable mode from the checked-out repository.

In [ ]:
%pip install -q -e rf-detr


Verify that Python imports RF-DETR from the cloned repository.

In [ ]:
import sys
from pathlib import Path

RFDETR_SRC = Path("rf-detr/src").resolve()
print("RFDETR_SRC:", RFDETR_SRC)
print("exists:", RFDETR_SRC.exists())

if str(RFDETR_SRC) not in sys.path:
    sys.path.insert(0, str(RFDETR_SRC))

import rfdetr

print("RF-DETR import OK")
print(Path(rfdetr.__file__).resolve())

<a name="runtime-class-injection"></a>

## 🧩 Runtime Class Injection

RF-DETR can be exported to ONNX, but some internal PyTorch module implementations create graph patterns that are difficult for the ONNX → RVC4 conversion path.

There are two places we make more export-friendly:

* **ADJUSTMENT 1**: the windowed DINOv2 backbone, where image windows are merged back into a full spatial feature map;
* **ADJUSTMENT 2**: `MSDeformAttn`, where multi-scale deformable attention computes sampling locations and attention weights.

The helper code below marks both changes with `# ADJUSTMENT 1` and `# ADJUSTMENT 2` comments, so you can quickly search for the modified graph patterns.

The same graph adjustment approach can also be used for other RF-DETR variants, such as RF-DETR Small, as long as the corresponding modules are present in the model.

The goal is not to change the model weights or train a different model. Instead, we express the same computation in a form that is easier for the RVC4 conversion path to handle.

This notebook writes a small local helper module and uses it to switch selected modules on the live `RFDETRNano` object before export:

```python
module.__class__ = RVC4MSDeformAttn
```

The helper cell is marked as hidden/collapsed in Colab because it is implementation-heavy. Depending on your notebook viewer, it may still appear expanded.



In [ ]:
# @title Run this cell to use the RVC4 class-injection helper
%%writefile rvc4_class_injection.py
"""Runtime class injection helpers for RF-DETR Nano -> RVC4 export.

This module intentionally does not modify RF-DETR source files.  It switches the
live RF-DETR PyTorch modules to subclasses whose ``forward`` methods reproduce
the RVC4-compatible module changes:

- ``WindowedDinov2WithRegistersBackbone``: avoid >4D transpose while undoing
  DINOv2 windowing.
- ``MSDeformAttn``: keep sampling locations flattened as
  ``[B, Len_q, Heads, Levels * Points, 2]`` instead of building rank-6 tensors.

Use with RF-DETR 1.7.1 before calling ``RFDETR.export(...)``.
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Optional

import torch
import torch.nn.functional as F  # noqa: N812
from torch import nn
from transformers.modeling_outputs import BackboneOutput

from rfdetr.models.backbone.dinov2_with_windowed_attn import (
    WindowedDinov2WithRegistersBackbone,
)
from rfdetr.models.ops.modules.ms_deform_attn import MSDeformAttn
from rfdetr.utilities.tensors import _bilinear_grid_sample


@dataclass(frozen=True)
class InjectionResult:
    """Summary of the modules switched by ``apply_rvc4_class_injection``."""

    windowed_backbone: list[str]
    ms_deform_attn: list[str]
    already_windowed_backbone: list[str]
    already_ms_deform_attn: list[str]

    @property
    def total_switched(self) -> int:
        return len(self.windowed_backbone) + len(self.ms_deform_attn)

    @property
    def total_already(self) -> int:
        return len(self.already_windowed_backbone) + len(self.already_ms_deform_attn)


def _merge_windowed_hidden_state(
    hidden_state: torch.Tensor,
    *,
    batch_size: int,
    num_windows: int,
    num_h_patches_per_window: int,
    num_w_patches_per_window: int,
) -> torch.Tensor:
    """Reconstruct spatial patch order without >4D transpose/permute ops."""

    # ADJUSTMENT 1: Windowed DINOv2 merge
    #
    # Helper replaces the original window unmerge pattern, which used
    # multiple high-rank reshapes followed by a 5D permute:
    #
    #     batch_windows, tokens_per_window, channels = hidden_state.shape
    #     hidden_state = hidden_state.reshape(
    #         batch_windows // num_windows_squared,
    #         num_windows_squared * tokens_per_window,
    #         channels,
    #     )
    #     hidden_state = hidden_state.reshape(
    #         (batch_windows // num_windows_squared) * num_windows,
    #         num_windows,
    #         num_h_patches_per_window,
    #         num_w_patches_per_window,
    #         channels,
    #     )
    #     hidden_state = hidden_state.permute(0, 2, 1, 3, 4)
    #
    # RVC4-compatible version:
    # keep the 6D reshape only for indexing, then rebuild the spatial grid by
    # concatenating windows row-by-row instead of using the high-rank permute.

    channels = hidden_state.shape[-1]
    hidden_state = hidden_state.reshape(
        batch_size,
        num_windows,
        num_windows,
        num_h_patches_per_window,
        num_w_patches_per_window,
        channels,
    )

    merged_rows = []
    for row_idx in range(num_windows):
        row_windows = [hidden_state[:, row_idx, col_idx] for col_idx in range(num_windows)]
        merged_rows.append(torch.cat(row_windows, dim=2))

    return torch.cat(merged_rows, dim=1)


class RVC4WindowedDinov2WithRegistersBackbone(WindowedDinov2WithRegistersBackbone):
    """RVC4-export-compatible DINOv2 windowed backbone.

    This is the RF-DETR 1.7.1 ``WindowedDinov2WithRegistersBackbone.forward``
    with only the window-merge block changed to use an RVC4-compatible merge.
    """

    def forward(
        self,
        pixel_values: torch.Tensor,
        output_hidden_states: Optional[bool] = None,
        output_attentions: Optional[bool] = None,
        return_dict: Optional[bool] = None,
    ) -> BackboneOutput:
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        output_hidden_states = (
            output_hidden_states if output_hidden_states is not None else self.config.output_hidden_states
        )
        output_attentions = output_attentions if output_attentions is not None else self.config.output_attentions

        embedding_output = self.embeddings(pixel_values)

        outputs = self.encoder(
            embedding_output,
            output_hidden_states=True,
            output_attentions=output_attentions,
            return_dict=return_dict,
        )

        hidden_states = outputs.hidden_states if return_dict else outputs[1]

        feature_maps = ()
        for stage, hidden_state in zip(self.stage_names, hidden_states):
            if stage in self.out_features:
                if self.config.apply_layernorm:
                    hidden_state = self.layernorm(hidden_state)
                if self.config.reshape_hidden_states:
                    hidden_state = hidden_state[:, self.num_register_tokens + 1 :]

                    # This keeps the original RF-DETR convention/bug noted in the
                    # source: normally this order would be height, width.
                    batch_size, _, height, width = pixel_values.shape
                    patch_size = self.config.patch_size

                    num_h_patches = height // patch_size
                    num_w_patches = width // patch_size

                    if self.config.num_windows > 1:
                        num_windows_squared = self.config.num_windows**2
                        batch_windows = hidden_state.shape[0]
                        num_h_patches_per_window = num_h_patches // self.config.num_windows
                        num_w_patches_per_window = num_w_patches // self.config.num_windows
                        hidden_state = _merge_windowed_hidden_state(
                            hidden_state,
                            batch_size=batch_windows // num_windows_squared,
                            num_windows=self.config.num_windows,
                            num_h_patches_per_window=num_h_patches_per_window,
                            num_w_patches_per_window=num_w_patches_per_window,
                        )

                    hidden_state = hidden_state.reshape(batch_size, num_h_patches, num_w_patches, -1)
                    hidden_state = hidden_state.permute(0, 3, 1, 2).contiguous()

                feature_maps += (hidden_state,)

        if not return_dict:
            if output_hidden_states:
                output = (feature_maps,) + outputs[1:]
            else:
                output = (feature_maps,) + outputs[2:]
            return output

        return BackboneOutput(
            feature_maps=feature_maps,
            hidden_states=outputs.hidden_states if output_hidden_states else None,
            attentions=outputs.attentions if output_attentions else None,
        )


def _flatten_sampling_locations(
    sampling_offsets: torch.Tensor,
    reference_points: torch.Tensor,
    input_spatial_shapes: torch.Tensor,
    n_points: int,
) -> torch.Tensor:
    """Build export-friendly sampling locations without rank-6 tensors."""

    # ADJUSTMENT 2: MSDeformAttn flattened sampling layout
    #
    # Helper avoids building sampling offsets/locations with separate
    # Levels and Points dimensions:
    #
    #     sampling_offsets = sampling_offsets.view(
    #         batch_size, len_query, n_heads, n_levels, n_points, 2
    #     )
    #     sampling_locations = (
    #         reference_points[:, :, None, :, None, :]
    #         + sampling_offsets / offset_normalizer[None, None, None, :, None, :]
    #     )
    #
    # RVC4-compatible version:
    # keep Levels * Points flattened:
    #
    #     sampling_offsets:   [B, Len_q, Heads, Levels * Points, 2]
    #     sampling_locations: [B, Len_q, Heads, Levels * Points, 2]
    #
    # This avoids unsupported high-rank tensor patterns in the ONNX -> SNPE/RVC4
    # conversion path.

    offset_normalizer = torch.stack([input_spatial_shapes[..., 1], input_spatial_shapes[..., 0]], -1)
    offset_normalizer = offset_normalizer[:, None, :].expand(-1, n_points, -1).reshape(-1, 2)

    if reference_points.shape[-1] == 2:
        reference_points_flat = reference_points.repeat_interleave(n_points, dim=2)[:, :, None, :, :]
        return reference_points_flat + sampling_offsets / offset_normalizer[None, None, None, :, :]

    if reference_points.shape[-1] == 4:
        reference_points_xy = reference_points[..., :2].repeat_interleave(n_points, dim=2)[:, :, None, :, :]
        reference_points_wh = reference_points[..., 2:].repeat_interleave(n_points, dim=2)[:, :, None, :, :]
        return reference_points_xy + sampling_offsets / n_points * reference_points_wh * 0.5

    raise ValueError(
        "Last dim of reference_points must be 2 or 4, "
        f"but got {reference_points.shape[-1]} instead."
    )


def rvc4_ms_deform_attn_core_pytorch(
    value: torch.Tensor,
    value_spatial_shapes: torch.Tensor,
    sampling_locations: torch.Tensor,
    attention_weights: torch.Tensor,
    value_spatial_shapes_hw: list[tuple[int, int]] | None = None,
) -> torch.Tensor:
    """MSDeformAttn core op using flattened sampling point layout.

    Accepts the RVC4-friendly 5D layout:
        sampling_locations: [B, Len_q, Heads, Levels * Points, 2]
        attention_weights:  [B, Len_q, Heads, Levels * Points]

    For convenience during local testing, it also accepts the upstream 6D/5D
    pair and flattens them internally.
    """

    # ADJUSTMENT 2: consume the flattened Levels * Points layout
    #
    # This core function is the runtime counterpart of the flattened MSDeformAttn
    # layout above. It consumes:
    #
    #     sampling_locations: [B, Len_q, Heads, Levels * Points, 2]
    #     attention_weights:  [B, Len_q, Heads, Levels * Points]
    #
    # and slices the flattened point dimension per feature level before grid
    # sampling.

    if sampling_locations.dim() == 6:
        batch_size_l, len_query_l, n_heads_l, num_levels_l, num_points_l, xy = sampling_locations.shape
        if xy != 2:
            raise ValueError(f"sampling_locations last dimension must be 2, got {xy}")
        sampling_locations = sampling_locations.reshape(
            batch_size_l,
            len_query_l,
            n_heads_l,
            num_levels_l * num_points_l,
            2,
        )

    if attention_weights.dim() == 5:
        batch_size_w, len_query_w, n_heads_w, num_levels_w, num_points_w = attention_weights.shape
        attention_weights = attention_weights.reshape(
            batch_size_w,
            len_query_w,
            n_heads_w,
            num_levels_w * num_points_w,
        )

    if sampling_locations.dim() != 5:
        raise ValueError(
            "sampling_locations must be rank 5 flattened or rank 6 upstream layout; "
            f"got shape {tuple(sampling_locations.shape)}"
        )
    if attention_weights.dim() != 4:
        raise ValueError(
            "attention_weights must be rank 4 flattened or rank 5 upstream layout; "
            f"got shape {tuple(attention_weights.shape)}"
        )

    batch_size, n_heads, head_dim, _ = value.shape
    shapes = value_spatial_shapes_hw if value_spatial_shapes_hw is not None else value_spatial_shapes
    num_levels = len(shapes)
    _, len_query, n_heads_from_locations, total_points, _ = sampling_locations.shape
    if n_heads_from_locations != n_heads:
        raise ValueError(f"n_heads mismatch: value has {n_heads}, sampling_locations has {n_heads_from_locations}")
    if total_points % num_levels != 0:
        raise ValueError(f"total_points={total_points} is not divisible by num_levels={num_levels}")
    num_points = total_points // num_levels

    sampling_grids = 2 * sampling_locations - 1
    level_sampling_grids = [
        sampling_grids[:, :, :, level_index * num_points : (level_index + 1) * num_points]
        .transpose(1, 2)
        .flatten(0, 1)
        for level_index in range(num_levels)
    ]

    value_list = value.split([height * width for height, width in shapes], dim=3)
    sampling_value_list = []
    for level_index, (height, width) in enumerate(shapes):
        value_l_ = value_list[level_index].view(batch_size * n_heads, head_dim, height, width)
        sampling_grid_l_ = level_sampling_grids[level_index]
        sampling_value_l_ = _bilinear_grid_sample(
            value_l_,
            sampling_grid_l_,
            padding_mode="zeros",
            align_corners=False,
        )
        sampling_value_list.append(sampling_value_l_)

    attention_weights = attention_weights.transpose(1, 2).reshape(
        batch_size * n_heads,
        1,
        len_query,
        num_levels * num_points,
    )
    sampling_value_list = torch.stack(sampling_value_list, dim=-2).flatten(-2)
    output = (sampling_value_list * attention_weights).sum(-1).view(batch_size, n_heads * head_dim, len_query)
    return output.transpose(1, 2).contiguous()


class RVC4MSDeformAttn(MSDeformAttn):
    """RVC4-export-compatible MSDeformAttn module."""

    def forward(
        self,
        query: torch.Tensor,
        reference_points: torch.Tensor,
        input_flatten: torch.Tensor,
        input_spatial_shapes: torch.Tensor,
        input_level_start_index: torch.Tensor,
        input_padding_mask: torch.Tensor | None = None,
        input_spatial_shapes_hw: list[tuple[int, int]] | None = None,
    ) -> torch.Tensor:
        batch_size, len_query, _ = query.shape
        batch_size, len_input, _ = input_flatten.shape
        expected_len_in = (input_spatial_shapes[:, 0] * input_spatial_shapes[:, 1]).sum()
        error_msg = "input_spatial_shapes must match the flattened input length"
        if getattr(self, "_export", False):
            torch._assert(expected_len_in == len_input, error_msg)
        else:
            assert expected_len_in == len_input, error_msg

        value = self.value_proj(input_flatten)
        if input_padding_mask is not None:
            value = value.masked_fill(input_padding_mask[..., None], float(0))

        # ADJUSTMENT 2: keep Levels * Points flattened instead of creating
        # [B, Len_q, Heads, Levels, Points, 2] sampling offsets.

        sampling_offsets = self.sampling_offsets(query).view(
            batch_size,
            len_query,
            self.n_heads,
            self.n_levels * self.n_points,
            2,
        )
        attention_weights = self.attention_weights(query).view(
            batch_size,
            len_query,
            self.n_heads,
            self.n_levels * self.n_points,
        )

        sampling_locations = _flatten_sampling_locations(
            sampling_offsets,
            reference_points,
            input_spatial_shapes,
            self.n_points,
        )
        attention_weights = F.softmax(attention_weights, -1)

        value = value.transpose(1, 2).contiguous().view(
            batch_size,
            self.n_heads,
            self.d_model // self.n_heads,
            len_input,
        )
        output = rvc4_ms_deform_attn_core_pytorch(
            value,
            input_spatial_shapes,
            sampling_locations,
            attention_weights,
            value_spatial_shapes_hw=input_spatial_shapes_hw,
        )
        output = self.output_proj(output)
        return output


def _resolve_torch_model(model_or_wrapper: Any) -> nn.Module:
    """Resolve RFDETR wrapper / ModelContext / nn.Module to the inner PyTorch model."""
    if isinstance(model_or_wrapper, nn.Module):
        return model_or_wrapper

    maybe_context = getattr(model_or_wrapper, "model", None)
    if isinstance(maybe_context, nn.Module):
        return maybe_context

    maybe_inner = getattr(maybe_context, "model", None)
    if isinstance(maybe_inner, nn.Module):
        return maybe_inner

    raise TypeError(
        "Could not resolve a torch.nn.Module. Pass RFDETRNano(), its ModelContext, "
        "or the inner LWDETR nn.Module."
    )


def apply_rvc4_class_injection(
    model_or_wrapper: Any,
    *,
    require: bool = True,
    verbose: bool = True,
) -> InjectionResult:
    """Switch live RF-DETR modules to RVC4-compatible subclasses.

    Args:
        model_or_wrapper: ``RFDETRNano()``, ``RFDETRNano().model`` or
            ``RFDETRNano().model.model``.
        require: If true, require exactly one backbone and at least one
            MSDeformAttn to be switched/already present.
        verbose: Print a small summary.
    """
    inner = _resolve_torch_model(model_or_wrapper)

    switched_windowed: list[str] = []
    switched_ms: list[str] = []
    already_windowed: list[str] = []
    already_ms: list[str] = []

    for name, module in inner.named_modules():
        if module.__class__ is RVC4WindowedDinov2WithRegistersBackbone:
            already_windowed.append(name)
        elif module.__class__ is WindowedDinov2WithRegistersBackbone:
            module.__class__ = RVC4WindowedDinov2WithRegistersBackbone
            switched_windowed.append(name)
        elif module.__class__ is RVC4MSDeformAttn:
            already_ms.append(name)
        elif module.__class__ is MSDeformAttn:
            module.__class__ = RVC4MSDeformAttn
            switched_ms.append(name)

    result = InjectionResult(
        windowed_backbone=switched_windowed,
        ms_deform_attn=switched_ms,
        already_windowed_backbone=already_windowed,
        already_ms_deform_attn=already_ms,
    )

    if require:
        total_windowed = len(result.windowed_backbone) + len(result.already_windowed_backbone)
        total_ms = len(result.ms_deform_attn) + len(result.already_ms_deform_attn)
        if total_windowed != 1:
            raise RuntimeError(f"Expected exactly 1 WindowedDinov2 backbone, found {total_windowed}: {result}")
        if total_ms < 1:
            raise RuntimeError(f"Expected at least 1 MSDeformAttn module, found {total_ms}: {result}")

    if verbose:
        print("RVC4 class injection result:")
        for name in result.windowed_backbone:
            print(f"  switched backbone: {name}")
        for name in result.ms_deform_attn:
            print(f"  switched MSDeformAttn: {name}")
        for name in result.already_windowed_backbone:
            print(f"  already RVC4 backbone: {name}")
        for name in result.already_ms_deform_attn:
            print(f"  already RVC4 MSDeformAttn: {name}")

    return result


def assert_rvc4_class_injection(model_or_wrapper: Any) -> None:
    """Raise if original target modules are still present after injection."""
    inner = _resolve_torch_model(model_or_wrapper)
    remaining: list[tuple[str, str]] = []
    found_rvc4: list[tuple[str, str]] = []

    for name, module in inner.named_modules():
        if module.__class__ is WindowedDinov2WithRegistersBackbone:
            remaining.append((name, module.__class__.__name__))
        elif module.__class__ is MSDeformAttn:
            remaining.append((name, module.__class__.__name__))
        elif module.__class__ in {RVC4WindowedDinov2WithRegistersBackbone, RVC4MSDeformAttn}:
            found_rvc4.append((name, module.__class__.__name__))

    if remaining:
        details = "\n".join(f"  {name}: {cls}" for name, cls in remaining)
        raise RuntimeError(f"Original export-sensitive modules remain:\n{details}")
    if not found_rvc4:
        raise RuntimeError("No RVC4-injected modules found.")


__all__ = [
    "InjectionResult",
    "RVC4MSDeformAttn",
    "RVC4WindowedDinov2WithRegistersBackbone",
    "apply_rvc4_class_injection",
    "assert_rvc4_class_injection",
    "rvc4_ms_deform_attn_core_pytorch",
]


<a name="export-rfdetr-to-onnx"></a>

## 📤 Export RF-DETR to ONNX

Now instantiate RF-DETR, apply runtime class injection, verify that the target modules were switched, and export the model to ONNX.

The expected module switch summary for RF-DETR Nano `1.7.1` is:

```text
switched backbone: backbone.0.encoder.encoder
switched MSDeformAttn: transformer.decoder.layers.0.cross_attn
switched MSDeformAttn: transformer.decoder.layers.1.cross_attn
```

In [ ]:
import sys
from pathlib import Path
import warnings
import torch

# Keep the notebook output focused. These warnings are expected for fixed-shape ONNX tracing.
warnings.filterwarnings("ignore", category=torch.jit.TracerWarning)

ROOT = Path.cwd()
ONNX_DIR = ROOT / "onnx_model"
ONNX_DIR.mkdir(exist_ok=True)

# Make the notebook-created helper module importable.
sys.path.insert(0, str(ROOT))

from rfdetr import RFDETRNano
from rvc4_class_injection import apply_rvc4_class_injection, assert_rvc4_class_injection

model = RFDETRNano()

result = apply_rvc4_class_injection(model, require=True, verbose=True)
assert_rvc4_class_injection(model)

print("Injection summary:", result)
print("Exporting ONNX to:", ONNX_DIR)

exported_path = model.export(
    output_dir=str(ONNX_DIR),
    shape=(384, 384),
    batch_size=1,
    dynamic_batch=False,
    opset_version=17,
    format="onnx",
)

print("Export result:", exported_path)


<a name="inspect-the-onnx-model"></a>

## 🔍 Inspect the ONNX Model (Optional)

We can validate the exported ONNX model and inspect the input/output interface.

In [ ]:
# @title Run this cell to make the inspection
from pathlib import Path
import onnx

onnx_path = Path("onnx_model") / "rfdetr-nano.onnx"

model = onnx.load(str(onnx_path))
onnx.checker.check_model(model)

print("ONNX path:", onnx_path)
print("IR version:", model.ir_version)
print("Opsets:", [opset.version for opset in model.opset_import])
print("Nodes:", len(model.graph.node))
print("Initializers:", len(model.graph.initializer))

print("Inputs:")
for value in model.graph.input:
    tensor_type = value.type.tensor_type
    shape = [
        dim.dim_value if dim.dim_value else dim.dim_param
        for dim in tensor_type.shape.dim
    ]
    print(f"- {value.name}: dtype={tensor_type.elem_type}, shape={shape}")

print("Outputs:")
for value in model.graph.output:
    tensor_type = value.type.tensor_type
    shape = [
        dim.dim_value if dim.dim_value else dim.dim_param
        for dim in tensor_type.shape.dim
    ]
    print(f"- {value.name}: dtype={tensor_type.elem_type}, shape={shape}")


Expected interface:

| Tensor | Shape | Description |
| --- | ---: | --- |
| `input` | `[1, 3, 384, 384]` | Input image tensor in NCHW layout |
| `dets` | `[1, 300, 4]` | Predicted boxes |
| `labels` | `[1, 300, 91]` | Class logits |
| opset | `17` | ONNX opset |

<a name="create-the-nn-archive"></a>

## 📦 Create the NN Archive

An NN Archive packages the model file together with the metadata required by DepthAI and HubAI, such as input layout, preprocessing, output tensors, and parser configuration.

In this step, we create the NN Archive from scratch with ArchiveGenerator. The archive will contain the exported ONNX model and the metadata needed to run RF-DETR Nano with the RFDETRParser.


In [ ]:
from pathlib import Path

from luxonis_ml.nn_archive.archive_generator import ArchiveGenerator
from luxonis_ml.nn_archive.config import CONFIG_VERSION
from rfdetr.assets.coco_classes import COCO_CLASSES, COCO_CLASS_NAMES

ONNX_PATH = Path("onnx_model") / "rfdetr-nano.onnx"
OUT_DIR = Path.cwd()
RFDETR_COCO_CLASSES = ["__unused__"] * 91
RFDETR_COCO_CLASSES[0] = "__background__"

assert len(COCO_CLASSES) == len(COCO_CLASS_NAMES)

for coco_id, class_name in zip(COCO_CLASSES, COCO_CLASS_NAMES):
    RFDETR_COCO_CLASSES[coco_id] = class_name
cfg_dict = {
    "config_version": CONFIG_VERSION,
    "model": {
        "metadata": {
            "name": "rfdetr-nano",
            "path": "rfdetr-nano.onnx",
            "precision": "float32",
        },
        "inputs": [
            {
                "name": "input",
                "dtype": "float32",
                "input_type": "image",
                "shape": [1, 3, 384, 384],
                "layout": "NCHW",
                "preprocessing": {
                    "mean": [123.675, 116.28, 103.53],
                    "scale": [58.395, 57.12, 57.375],
                    "reverse_channels": None,
                    "interleaved_to_planar": None,
                    "dai_type": "RGB888p",
                },
            }
        ],
        "outputs": [
            {
                "name": "dets",
                "dtype": "float32",
                "shape": [1, 300, 4],
                "layout": "NCD",
            },
            {
                "name": "labels",
                "dtype": "float32",
                "shape": [1, 300, 91],
                "layout": "NCD",
            },
        ],
        "heads": [
            {
                "parser": "RFDETRParser",
                "outputs": ["dets", "labels"],
                "metadata": {
                    "classes": RFDETR_COCO_CLASSES,
                    "n_classes": 91,
                    "conf_threshold": 0.5,
                    "max_det": 300,
                },
            }
        ],
    },
}

generator = ArchiveGenerator(
    archive_name="rfdetr-nano-onnx",
    save_path=str(OUT_DIR),
    cfg_dict=cfg_dict,
    executables_paths=[str(ONNX_PATH)],
)

archive_path = generator.make_archive()
print(f"Created NN Archive: {archive_path}")

head_metadata = cfg_dict["model"]["heads"][0]["metadata"]

assert "classes" in head_metadata
assert len(head_metadata["classes"]) == head_metadata["n_classes"]

print("Parser:", cfg_dict["model"]["heads"][0]["parser"])
print("Number of classes:", len(head_metadata["classes"]))
print("Sample classes:")
for class_id in [1, 2, 3, 17, 90]:
    print(f"  {class_id}: {head_metadata['classes'][class_id]}")


<a name="convert-the-nn-archive-to-rvc4"></a>

## 🤖 Convert the NN Archive to RVC4

Now convert the NN Archive to an RVC4 archive with HubAI.

This tutorial uses HubAI RVC4 conversion with `FP16_STANDARD` precision and SNPE `2.41.0`.

INT8 precision is not used because INT8 conversion for this model is not currently supported by the Qualcomm toolchain path used for RVC4.

You can get your HubAI API key from https://hub.luxonis.com/team-settings/api-keys.

This step runs the RVC4 conversion through HubAI and may take several minutes.

In [ ]:
import os
import getpass

if not os.environ.get("HUBAI_API_KEY"):
    os.environ["HUBAI_API_KEY"] = getpass.getpass("Enter your HubAI API key: ")

print("HUBAI_API_KEY is set:", bool(os.environ.get("HUBAI_API_KEY")))

In [ ]:
import os
from pathlib import Path

from hubai_sdk import HubAIClient

ARCHIVE_PATH = Path("rfdetr-nano-onnx.tar.xz")
OUT_DIR = Path.cwd()

client = HubAIClient(api_key=os.environ["HUBAI_API_KEY"])

response = client.convert.RVC4(
    path=str(ARCHIVE_PATH),
    name="rfdetr-nano-rvc4",
    quantization_mode="FP16_STANDARD",
    tool_version="2.41.0",
    output_dir=str(OUT_DIR),
)

print(f"Downloaded path: {response.downloaded_path}")


<a name="depthai-script"></a>

## 📷 DepthAI Script

This section requires DepthAI v3, DepthAI Nodes, and a Luxonis RVC4 device connected to the machine running the notebook.

To run the model on a DepthAI device using the script below, please note the following:

* You can view the output stream by opening http://localhost:8082 in your browser.
* If you're running the script from a Jupyter Notebook, the output may not appear directly within the notebook. The script should print a link pointing to http://localhost:8082 for accessing the stream.
* To stop the video stream, press q while focused on the visualizer page.

In [ ]:
from pathlib import Path

import depthai as dai
from depthai_nodes.node.parsing_neural_network import ParsingNeuralNetwork

DEVICE = None  # Set to None to use the default device, or specify an RVC4 device IP/MXID.
MODEL_PATH = Path("rfdetr-nano-onnx.rvc4.tar.xz")
FPS = 10

available_devices = dai.Device.getAllAvailableDevices()

if DEVICE is None and not available_devices:
    raise RuntimeError(
        "No DepthAI devices found. Connect a Luxonis RVC4 device before running this cell."
    )

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"RVC4 archive not found: {MODEL_PATH}. Make sure the conversion step completed successfully."
    )

device = dai.Device(dai.DeviceInfo(DEVICE)) if DEVICE else dai.Device()
platform = device.getPlatform().name

if platform != "RVC4":
    raise RuntimeError(
        f"This tutorial requires a Luxonis RVC4 device, but the connected device platform is {platform}."
    )

visualizer = dai.RemoteConnection(httpPort=8082)
nn_archive = dai.NNArchive(str(MODEL_PATH))

with dai.Pipeline(device) as pipeline:
    cam = pipeline.create(dai.node.Camera).build(sensorFps=FPS)
    cam_out = cam.requestOutput(
        size=(384, 384),
        fps=FPS,
        type=dai.ImgFrame.Type.BGR888i,
    )

    nn_with_parser = pipeline.create(ParsingNeuralNetwork).build(
        input=cam_out,
        nn_source=nn_archive,
    )

    visualizer.addTopic(topicName="rgb", output=nn_with_parser.passthrough)
    visualizer.addTopic(topicName="detections", output=nn_with_parser.out)

    pipeline.start()
    visualizer.registerPipeline(pipeline)

    print("Open http://localhost:8082 in your browser.")
    print("Press q in the visualizer window to stop.")

    while pipeline.isRunning():
        pipeline.processTasks()
        key = visualizer.waitKey(1)
        if key == ord("q"):
            print("Stopping pipeline.")
            break

<a name="summary"></a>

## 🎉 Summary

Yay! 🎉 Huge congratulations, you have successfully converted RF-DETR to RVC4.

The flow was:

```text
RF-DETR Nano
→ runtime class injection before ONNX export
→ ONNX model
→ NN Archive created with ArchiveGenerator
→ RVC4 conversion with HubAI
→ DepthAI runtime path
```

The main output file is:

```text
rfdetr-nano-onnx.rvc4.tar.xz
```
